WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import pathlib

import imageio.v2 as imageio
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Wedge

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)

figure_dir = pathlib.Path("crops/notebooks/dev/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
rod_length = 0.100  # [m]
inner_radius = 0.0125  # [m]
outer_radius = 0.0155  # [m]
mid_radius = 0.5 * (inner_radius + outer_radius)

num_elems = 120
num_nodes = num_elems + 1
num_fibers = 96  # sectors around the wall; one ring only
parametric_coords = jnp.linspace(0.0, rod_length, num_nodes)

sector_edges = np.linspace(0.0, 2.0 * np.pi, num_fibers + 1)
sector_start, sector_end = sector_edges[:-1], sector_edges[1:]
sector_inner = np.full(num_fibers, inner_radius)
sector_outer = np.full(num_fibers, outer_radius)

# Closed-form area, centroid and centroidal second moments of an annular sector.
span = sector_end - sector_start
sector_areas = 0.5 * span * (sector_outer**2 - sector_inner**2)
cubic = (sector_outer**3 - sector_inner**3) / 3.0
centroid_d1 = cubic * (np.sin(sector_end) - np.sin(sector_start)) / sector_areas
centroid_d2 = cubic * (np.cos(sector_start) - np.cos(sector_end)) / sector_areas
quartic = (sector_outer**4 - sector_inner**4) / 4.0
double_angle = 0.25 * (np.sin(2.0 * sector_end) - np.sin(2.0 * sector_start))
moi_about_d1 = quartic * (0.5 * span - double_angle) - sector_areas * centroid_d2**2
moi_about_d2 = quartic * (0.5 * span + double_angle) - sector_areas * centroid_d1**2


def along_rod(values: np.ndarray) -> np.ndarray:
  """Repeats a per-fiber quantity over every element of the rod.

  Args:
    values: Array of shape `(num_fibers, ...)` holding one entry per fiber.

  Returns:
    Array of shape `(num_elems, num_fibers, ...)` with the input broadcast along
    the axial direction.
  """
  return np.broadcast_to(values, (num_elems,) + np.shape(values))


section = dict(
  fiber_inner_radii=along_rod(sector_inner),
  fiber_outer_radii=along_rod(sector_outer),
  fiber_start_angles=along_rod(sector_start),
  fiber_end_angles=along_rod(sector_end),
)

elem_axial = np.asarray(0.5 * (parametric_coords[:-1] + parametric_coords[1:]))  # (e,)
sector_theta = np.arctan2(centroid_d2, centroid_d1) % (2.0 * np.pi)  # (f,)
axial_grid, theta_grid = np.meshgrid(elem_axial, sector_theta, indexing="ij")

print(
  f"{num_elems * num_fibers} cells, wall {1e3 * (outer_radius - inner_radius):.1f} mm"
)
print(f"sector arc at mid radius = {1e3 * mid_radius * span[0]:.2f} mm")
print(f"element length = {1e3 * rod_length / num_elems:.2f} mm")

In [ ]:
helix_angle = np.radians(-45.0)  # alpha, peaks the coupling at 45 degrees
num_starts = 6
slit_angular_fraction = 0.55  # fraction of each start's angular period that is cut
bridge_pitch = 0.020  # [m] axial period of the ligaments interrupting a cut
bridge_fraction = 0.18  # fraction of that period that stays solid
collar_length = 0.020  # [m] unpatterned length at each end

start_period = 2.0 * np.pi / num_starts
helix_phase = theta_grid - axial_grid * np.tan(helix_angle) / mid_radius
start_index = np.floor(helix_phase / start_period)
band_phase = np.mod(helix_phase, start_period) / start_period  # in [0, 1)

bridge_phase = np.mod(
  axial_grid / bridge_pitch + start_index / num_starts, 1.0
)  # staggered so bridges spiral instead of forming rings

in_collar = (axial_grid < collar_length) | (axial_grid > rod_length - collar_length)
is_cut = (band_phase < slit_angular_fraction) & (bridge_phase > bridge_fraction)
solid_mask = (~is_cut) | in_collar  # (e, f) True where material remains

winding_angles = np.where(in_collar, 0.0, helix_angle)  # (e, f)
patterned_length = rod_length - 2.0 * collar_length

print(f"open area fraction = {1.0 - solid_mask.mean():.3f}")
print(f"helix angle = {np.degrees(helix_angle):.0f} deg, {num_starts} starts")
print(
  f"patterned length = {1e3 * patterned_length:.0f} mm of {1e3 * rod_length:.0f} mm"
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

ax.imshow(
  solid_mask.T,
  origin="lower",
  aspect="auto",
  cmap="coolwarm",
  extent=(0.0, 1e3 * rod_length, 0.0, 360.0),
)
# ax.axvline(1e3 * collar_length, color="tab:red", lw=1.0)
# ax.axvline(1e3 * (rod_length - collar_length), color="tab:red", lw=1.0)
ax.set_xlabel("axial [mm]")
ax.set_ylabel(r"$\theta$ [deg]")

fig.tight_layout()
plt.show()

print("solid cells per section, min over elements =", solid_mask.sum(axis=1).min())

In [ ]:
shear_modulus = 5.0e6  # [Pa] TPU-like
void_fraction = 1.0e-4
reinforcement_modulus = 4.0 * shear_modulus  # eta_f; sweep this

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=parametric_coords,
  fiber_offsets=jnp.asarray(along_rod(np.stack([centroid_d1, centroid_d2], axis=-1))),
  fiber_areas=jnp.asarray(along_rod(sector_areas)),
  fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
  fiber_centroid_moi=jnp.asarray(
    along_rod(np.stack([moi_about_d1, moi_about_d2], axis=-1))
  ),
  fiber_winding_angles=jnp.asarray(winding_angles),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

solid = jnp.asarray(solid_mask.astype(float))
shear_moduli = shear_modulus * (solid + void_fraction * (1.0 - solid))
reinforced = jnp.asarray((solid_mask & ~in_collar).astype(float))
reinforcement_moduli = reinforcement_modulus * reinforced

print(f"G ranges {float(shear_moduli.min()):.3e} to {float(shear_moduli.max()):.3e} Pa")
print(f"reinforced cells = {float(reinforced.mean()):.2f} of all cells")

In [ ]:
total_twist = 2*np.pi  # half a turn; push higher and expect helical buckling
num_steps = 60
twist_ramp = np.linspace(total_twist / num_steps, total_twist, num_steps)

root, tip = np.array([0]), np.array([num_nodes - 1])
clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
tip_held_dofs = (
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)

boundary_list = (
  [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
      node_targets=[root for _ in range(num_steps)],
      name=f"root_{dof}",
    )
    for dof in clamped_dofs
  ]
  + [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
      node_targets=[tip for _ in range(num_steps)],
      name=f"tip_{dof}",
    )
    for dof in tip_held_dofs
  ]
  + [
    _bc.DirichletBC(
      values=(_utils.Rotation.THETA_X, [np.array([angle]) for angle in twist_ramp]),
      node_targets=[tip for _ in range(num_steps)],
      name="tip_twist",
    )
  ]
)

boundary_conditions = _bc.process_boundary_conditions(rod, boundary_list)


In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10},
)
problem = _fe.CosseratProblem(rod=rod, bc=boundary_conditions, nr_solver=nr_solver)

print(
  f"{num_steps} steps, {np.degrees(total_twist / num_steps / num_elems):.2f} deg "
  f"per element per step"
)

In [ ]:
def solve_case(reinforcement_field: jnp.ndarray) -> tuple[np.ndarray, np.ndarray]:
  """Twists the tube through the prescribed ramp and returns the dof history.

  Args:
    reinforcement_field: Array of shape `(num_elems, num_fibers)` [Pa] holding the
      reinforcement stiffness of every cell. An array of zeros gives the purely
      isotropic response.

  Returns:
    A tuple `(u_star, history)`. The first is the converged dof vector at full
    twist, of shape `(num_dofs,)`. The second has shape
    `(num_steps, num_nodes, num_dofs_per_node)`.
  """
  u_star, u_history = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=shear_moduli,
    reinforcement_moduli=reinforcement_field,
  )
  history = np.asarray(u_history).reshape(num_steps, num_nodes, -1)
  return np.asarray(u_star), history


In [ ]:
results = {
  # "isotropic": solve_case(jnp.zeros_like(reinforcement_moduli)),
  "reinforced": solve_case(reinforcement_moduli),
}

fig, (ax_ext, ax_ratio) = plt.subplots(1, 2, figsize=(11, 4))
twist_deg = np.degrees(twist_ramp)

for name, (u_star, history) in results.items():
  extension = history[:, -1, _utils.Translation.X.value]
  ax_ext.plot(twist_deg, 1e3 * extension, lw=1.6, label=name)
  ax_ratio.plot(twist_deg, extension / (mid_radius * twist_ramp), lw=1.6, label=name)

  strain = extension[-1] / patterned_length
  print(
    f"{name:12s} extension = {1e3 * extension[-1]:+7.3f} mm, "
    f"actuation strain = {100.0 * strain:+6.2f} %, "
    f"coupling ratio = {extension[-1] / (mid_radius * total_twist):+6.4f}"
  )

ax_ext.set_xlabel("prescribed tip rotation [deg]")
ax_ext.set_ylabel("tip extension [mm]")
ax_ext.legend()
ax_ratio.set_xlabel("prescribed tip rotation [deg]")
ax_ratio.set_ylabel(r"coupling ratio $\Delta L / (R\phi)$")
ax_ratio.legend()
fig.tight_layout()
plt.show()

In [ ]:
def draw_tube(coords: np.ndarray, directors: np.ndarray, ax: plt.Axes) -> plt.Axes:
  """Renders the slitted tube as annular-sector cells with the cuts left empty.

  Args:
    coords: Array of shape `(num_nodes, num_dim=3)` [m] of the centerline in the
      configuration to draw.
    directors: Array of shape `(num_nodes, num_dirs=3, num_dim=3)` of the material
      frames at those nodes.
    ax: Matplotlib 3D axis to draw onto.

  Returns:
    The same axis, with the viewport re-framed around the tube.
  """
  ax = _viz.plot_rod_annular_sections(
    node_coords=coords,
    directors=directors,
    **section,
    num_arc_segments=2,
    fiber_field=np.asarray(winding_angles),
    # fiber_field_val_range=(0.0, float(helix_angle)),
    fiber_mask=solid_mask,
    cmap="coolwarm",
    edge_width=0.015,
    ax=ax,
  )
  ax.set_xlim(-0.01, rod_length + 0.02)
  ax.set_ylim(-3.0 * outer_radius, 3.0 * outer_radius)
  ax.set_zlim(-3.0 * outer_radius, 3.0 * outer_radius)
  ax.view_init(elev=18.0, azim=-72.0)
  return ax


u_star, history = results["reinforced"]
fig = plt.figure(figsize=(12, 5))
ax_ref = fig.add_subplot(121, projection="3d")
draw_tube(np.asarray(rod.node_coords), np.asarray(rod.directors_0), ax_ref)
ax_ref.set_title("undeformed")

coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
  rod, jnp.asarray(u_star)
)
ax_def = fig.add_subplot(122, projection="3d")
draw_tube(np.asarray(coords), np.asarray(directors), ax_def)
ax_def.set_title(f"twisted {np.degrees(total_twist):.0f} deg")
fig.tight_layout()
plt.show()

In [ ]:
frame_steps = range(0, num_steps, max(1, num_steps // 10))
frame_paths = []
for step in frame_steps:
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod, jnp.asarray(history[step].ravel())
  )
  fig = plt.figure(figsize=(6, 5))
  ax = fig.add_subplot(111, projection="3d")
  draw_tube(np.asarray(coords), np.asarray(directors), ax)
  ax.set_title(f"{np.degrees(twist_ramp[step]):.0f} deg, "
               f"{1e3 * history[step, -1, _utils.Translation.X.value]:+.2f} mm")
  path = figure_dir / f"hsa_frame_{step:03d}.png"
  fig.savefig(path, dpi=110, bbox_inches="tight")
  plt.close(fig)
  frame_paths.append(path)

imageio.mimsave(
  figure_dir / "hsa_twist.gif",
  [imageio.imread(path) for path in frame_paths],
  duration=0.25,
  loop=0,
)
print(f"wrote {len(frame_paths)} frames to {figure_dir / 'hsa_twist.gif'}")